## C. Crawling Berita

In [ ]:
!pip install requests
!pip install beautifulsoup4
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [ ]:
!pip install builtwith

  Preparing metadata (setup.py) ... done
  Created wheel for builtwith: filename=builtwith-1.3.4-py3-none-any.whl size=36077 sha256=bbf93d04d625180933c22fd6c25e231a71f66995b5daff32fdacba6907a3addc
  Stored in directory: /root/.cache/pip/wheels/7f/2d/b2/606e3df914d4aeeab99c4a4e3e9a61673d2293c2e346db00c8
Successfully built builtwith


In [5]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urlparse

def scrape_kompas_article(url):
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    try:
        r = requests.get(url, headers=headers)
        r.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error saat mengambil URL: {e}")
        return None

    soup = BeautifulSoup(r.content, "html.parser")

    # Judul
    judul = soup.select_one("h1.read__title")
    judul_text = judul.text.strip() if judul else "Tidak ditemukan judul"

    # Isi (100 kata)
    isi_elems = soup.select("div.read__content p")
    isi_text = " ".join([p.get_text(strip=True) for p in isi_elems]) if isi_elems else "Tidak ditemukan isi"
    words = isi_text.split()
    isi_100 = " ".join(words[:100]) + ("..." if len(words) > 100 else "")

    # Cari kategori
    kategori_text = "Tidak ditemukan kategori"

    kategori_meta = soup.find("meta", {"property": "article:section"})
    if kategori_meta and kategori_meta.get("content"):
        kategori_text = kategori_meta["content"].strip()

    if kategori_text == "Tidak ditemukan kategori":
        breadcrumb = soup.select("div.breadcrumb__link a")
        if breadcrumb and len(breadcrumb) > 1:
            kategori_text = breadcrumb[1].get_text(strip=True)

    if kategori_text == "Tidak ditemukan kategori":
        parsed = urlparse(url)
        subdomain = parsed.netloc.split(".")[0]
        if subdomain and subdomain != "www" and subdomain != "kompas":
            kategori_text = subdomain.capitalize()

    return {
        "Judul": judul_text,
        "Isi (100 kata)": isi_100,
        "Kategori": kategori_text
    }

# Daftar URL
urls_to_test = [
    "https://nasional.kompas.com/read/2024/08/16/11404101/jokowi-kenakan-pakaian-adat-betawi-di-sidang-tahunan-terakhirnya-simbol",
    "https://nasional.kompas.com/read/2024/02/13/21583481/kpu-tegaskan-pemilih-tak-terdaftar-di-dpt-bisa-nyoblos-begini-mekanismenya",
    "https://megapolitan.kompas.com/read/2024/05/11/19123091/warga-sebut-ada-benda-serupa-jimat-pada-mayat-dalam-sarung-di-pamulang",
    "https://megapolitan.kompas.com/read/2024/08/26/07235981/polisi-menganiaya-mereka-yang-cinta-damai-dan-bercita-cita-mulia",
    "https://nasional.kompas.com/read/2024/08/12/08444611/airlangga-hartarto-mundur-dari-ketum-golkar-bantah-karena-korupsi-dan-fokus",
    "https://nasional.kompas.com/read/2024/07/29/17462851/nasdem-pastikan-dukung-ilham-habibie-untuk-pilkada-jabar",
    "https://megapolitan.kompas.com/read/2024/08/02/23392851/influencer-parenting-aniaya-balita-kriminolog-kemampuan-di-bidang",
    "https://nasional.kompas.com/read/2024/08/21/05232081/jalan-mulus-bahlil-raih-kursi-ketum-golkar",
"https://nasional.kompas.com/read/2024/01/19/22313101/maruarar-sirait-dukung-prabowo-gibran-tkn-mari-kerja-keras-bersama",
"https://megapolitan.kompas.com/read/2024/08/12/21025981/usai-diturap-kali-mati-di-pademangan-tak-lagi-jadi-tempat-berenang-anak",
"https://megapolitan.kompas.com/read/2024/08/29/12432031/maju-pilkada-depok-imam-budi-hartono-ingin-lanjutkan-program-walkot-idris",
"https://megapolitan.kompas.com/read/2024/08/28/13024071/kpud-kota-bekasi-terima-dokumen-pendaftaran-paslon-tri-adhianto-dan",
"https://nasional.kompas.com/read/2024/07/29/05000011/-populer-nasional-pbnu-heran-pansus-haji-dibentuk-muhadjir-effendy-jadi",
"https://nasional.kompas.com/read/2024/04/06/21590521/antoninho-rangel-da-silva-jenderal-berdarah-timor-leste-yang-kini-jabat",
"https://nasional.kompas.com/read/2024/09/04/12443721/pidato-lengkap-paus-fransiskus-di-istana-negara",
"https://nasional.kompas.com/read/2024/02/24/20074351/ahy-serahkan-kepada-prabowo-jika-ingin-ajak-parpol-pengusung-anies-bergabung",
"https://nasional.kompas.com/read/2024/08/06/20310821/wacana-kotak-kosong-pada-pilkada-jakarta-jadi-ironi-bagi-demokrasi",
"https://nasional.kompas.com/read/2024/06/09/20234841/cerita-korban-penipuan-visa-haji-ilegal-panas-dingin-takut-ditangkap-polisi",
"https://nasional.kompas.com/read/2024/08/20/07025071/gerindra-sebut-reshuffle-menkumham-untuk-sinkronisasi-pemerintahan-ke-depan",
"https://nasional.kompas.com/read/2024/01/09/22152121/puan-salami-anies-muhaimin-usai-debat-capres-pkb-seperti-taufiq-kiemas",
"https://megapolitan.kompas.com/read/2024/09/04/16151141/3-paslon-ramaikan-pilkada-jakarta-bamus-betawi-warga-punya-banyak-pilihan",
"https://megapolitan.kompas.com/read/2024/09/03/17275901/berita-foto-tangis-bahagia-warga-sambut-kedatangan-paus-fransiskus-di",
"https://nasional.kompas.com/read/2024/01/11/22261141/dilaporkan-ke-dewas-wakil-ketua-kpk-mengaku-tak-pernah-komunikasi-dengan",
"https://megapolitan.kompas.com/read/2024/07/26/19404751/pengamat-nilai-dharma-kun-bakal-sulit-dapat-538178-data-dukungan-selama",
"https://megapolitan.kompas.com/read/2024/06/21/22434391/kerap-dipandang-sebelah-mata-jadi-pelukis-jalanan-atu-bagi-saya-tidak",
"https://nasional.kompas.com/read/2024/08/27/16243201/pramono-anung-belum-final-pdi-p-masih-berpeluang-usung-anies-di-jakarta",
"https://megapolitan.kompas.com/read/2024/08/29/21383931/naik-becak-ke-kpu-uu-saeful-nurul-sumarheni-daftar-pilkada-bekasi",
"https://nasional.kompas.com/read/2024/08/24/10493841/publik-diminta-jangan-mau-dininabobokan-dpr-kpu-tetap-kawal-putusan-mk",
"https://megapolitan.kompas.com/read/2024/08/29/16534131/daftar-ke-kpu-supian-chandra-resmi-maju-pada-pilkada-depok-2024",
"https://nasional.kompas.com/read/2024/05/02/21391271/icw-sebut-alasan-nurul-ghufron-absen-di-sidang-etik-dewas-kpk-tak-bisa",
"https://nasional.kompas.com/read/2024/08/06/16442871/program-belanja-bareng-yatim-ajak-ratusan-anak-yatim-berbelanja-di-gps-mall",
"https://megapolitan.kompas.com/read/2024/08/28/09353381/didorong-pdi-p-jadi-bacagub-jakarta-pramono-berpikir-dan-berkeinginan",
"https://nasional.kompas.com/read/2024/05/21/20571641/kpk-sita-mobil-mercy-di-makassar-diduga-disembunyikan-syl",
"https://nasional.kompas.com/read/2024/07/31/13055941/pertemuan-prabowo-erdogan-bahas-serangan-israel-ke-palestina",
"https://nasional.kompas.com/read/2024/08/14/07180031/tinjau-layanan-publik-bpom-menpan-rb-dukung-penguatan-digitalisasi-layanan",
"https://megapolitan.kompas.com/read/2024/05/08/23142931/bantah-pernyataan-ketua-stip-soal-tak-ada-lagi-perpeloncoan-alumni-masih",
"https://nasional.kompas.com/read/2024/08/27/11582931/sindir-megawati-bahlil-bilang-tak-minta-pasangan-airin-jadi-kader-golkar",
"https://nasional.kompas.com/read/2024/09/02/19032291/anies-cerita-pernah-tawarkan-diri-jadi-kader-partai-tapi-malah-ditolak",
"https://nasional.kompas.com/read/2024/09/03/21352571/pada-iaf-2024-kepala-nfa-arief-prasetyo-adi-elaborasi-peran-aktif-indonesia",
"https://nasional.kompas.com/read/2024/05/26/22004211/sambut-pilkada-2024-megawati-minta-kader-pdip-turun-ke-akar-rumput",
"https://nasional.kompas.com/read/2024/04/13/18282131/opm-ajukan-syarat-pembebasan-pilot-susi-air-philips-mark-mehrtens",
"https://megapolitan.kompas.com/read/2024/07/28/09413881/ditanya-soal-rencana-program-khusus-perempuan-di-jakarta-anies-banyak",
"https://nasional.kompas.com/read/2024/08/08/11563851/menlu-retno-beri-saran-untuk-para-wni-yang-akan-bepergian-ke-inggris",
"https://megapolitan.kompas.com/read/2024/07/19/22430791/pelaku-minta-maaf-kasus-perempuan-direkam-diam-diam-di-krl-berakhir-damai",
"https://nasional.kompas.com/read/2024/07/29/13592631/pansus-haji-dpr-batal-gelar-rapat-pada-masa-reses",
"https://megapolitan.kompas.com/read/2024/08/07/16340031/satpol-pp-kota-bekasi-bakal-razia-warung-penjual-rokok-ilegal",
"https://nasional.kompas.com/read/2024/08/13/11204411/belanja-daerah-baru-31-persen-jokowi-kecil-sekali-segera-realisasikan",
"https://nasional.kompas.com/read/2024/08/01/17345351/keluarga-korban-penganiayaan-pemilik-daycare-di-depok-minta-asistensi",
"https://nasional.kompas.com/read/2024/08/01/20374201/berduka-atas-wafatnya-pemimpin-hamas-ismail-haniyeh-wapres-kita-kehilangan",
"https://megapolitan.kompas.com/read/2024/08/07/20140301/lukman-edy-tak-hanya-dilaporkan-ke-bareskrim-tapi-juga-ke-polres-kota",
"https://megapolitan.kompas.com/read/2024/08/07/20290361/polisi-akan-periksa-pengendara-ekskavator-yang-bikin-pipa-gas-bocor-di",
"https://nasional.kompas.com/read/2024/07/26/22500721/ditetapkan-jadi-tersangka-ujang-iskandar-miliki-harta-kekayaan-rp-187-miliar",
"https://megapolitan.kompas.com/read/2024/08/07/12330571/wali-kota-rumah-panggung-dari-kemenhan-di-muara-angke-dibangun-sesuai",
"https://megapolitan.kompas.com/read/2024/01/31/22472491/anggota-dprd-dki-ajak-warga-tanding-sepak-bola-di-jis-biar-bisa-rasakan",
"https://nasional.kompas.com/read/2024/08/16/09574431/megawati-tak-tampak-dalam-sidang-tahunan-mpr",
"https://nasional.kompas.com/read/2024/09/02/16014281/bareskrim-tangkap-penjual-video-porno-di-telegram-aksi-dilakukan-sejak-2022",
"https://nasional.kompas.com/read/2024/08/27/06154941/memaknai-pernyataan-prabowo-soal-haus-kekuasaan",
"https://megapolitan.kompas.com/read/2024/04/18/21212681/banyak-warga-menonton-kebakaran-toko-bingkai-lalin-di-simpang-mampang",
"https://megapolitan.kompas.com/read/2024/08/05/15340441/ridwan-kamil-otw-jakarta-peluang-lawan-kotak-kosong-disebut-sangat",
"https://nasional.kompas.com/read/2024/07/30/13411041/benny-rhamdani-bongkar-inisial-lain-diduga-terlibat-sindikat-judi-online",
"https://nasional.kompas.com/read/2024/09/05/11422681/menlu-belum-ada-temuan-wni-bentuk-geng-di-jepang",
"https://nasional.kompas.com/read/2024/08/25/13331801/ratusan-orang-sempat-protes-muktamar-pkb-cak-imin-kalau-kalian-kader-nu",
"https://megapolitan.kompas.com/read/2024/09/04/11351481/lambaikan-tangan-from-mobil-paus-fransiskus-sapa-warga-yang-menantinya-di",
"https://nasional.kompas.com/read/2024/08/28/14161421/polri-siapkan-pengamanan-kunjungan-paus-fransiskus-dan-kegiatan-isf",
"https://megapolitan.kompas.com/read/2024/08/30/10562021/pramono-anung-rano-karno-mengaku-rutin-periksa-kesehatan",
"https://nasional.kompas.com/read/2024/08/05/16015011/tak-usung-ridwan-kamil-pada-pilkada-jakarta-nasdem-konsisten-kalah-menang-ya",
"https://nasional.kompas.com/read/2024/07/25/18405201/menag-yaqut-perintahkan-bpjph-cek-label-halal-pada-roti-okko",
"https://megapolitan.kompas.com/read/2024/08/12/09515761/kbri-yangon-berupaya-selamatkan-wni-yang-diduga-disekap-dan-dianiaya-di",
"https://megapolitan.kompas.com/read/2024/08/23/17253291/giliran-itb-unpas-uninda-dan-unindra-demo-di-gedung-dpr-tolak-revisi-uu",
"https://nasional.kompas.com/read/2024/07/30/15355211/tni-tetap-aktifkan-posko-pengaduan-netralitas-prajurit-saat-pilkada",
"https://megapolitan.kompas.com/read/2024/05/17/22443071/pria-di-kali-sodong-dibunuh-debt-collector-gadungan-karena-tolak-serahkan",
"https://nasional.kompas.com/read/2024/01/13/20041561/pengancam-anies-ditangkap-tkn-minta-pendukung-prabowo-sampaikan-dukungan",
"https://megapolitan.kompas.com/read/2024/09/03/11525681/mary-dan-irfan-berikan-bunga-tangan-untuk-paus-fransiskus",
"https://megapolitan.kompas.com/read/2024/05/04/21072841/motif-pelaku-aniaya-taruna-stip-hingga-tewas-senioritas-dan-arogansi",
"https://nasional.kompas.com/read/2024/08/24/09325691/ppp-dukung-petahana-ansar-ahmad-dan-nyangnyang-haris-pada-pilkada-kepulauan",
"https://nasional.kompas.com/read/2024/08/19/17533431/pkb-anggap-anies-punya-peran-lain-selain-calon-gubernur-jakarta",
"https://nasional.kompas.com/read/2024/08/14/12420361/soal-peluang-bahlil-jadi-calon-tunggal-ketum-golkar-keguyuban-lebih-penting",
"https://nasional.kompas.com/read/2024/08/26/00000071/tanggal-29-agustus-2024-memperingati-hari-apa",
"https://nasional.kompas.com/read/2024/08/20/18265831/komisi-ii-bakal-bahas-putusan-mk-dengan-kpu-pekan-depan",
"https://megapolitan.kompas.com/read/2024/08/24/11092641/jaminan-dasco-dan-habiburokhman-demi-bebaskan-pedemo-tak-berlaku",
"https://megapolitan.kompas.com/read/2024/04/24/20504291/kajari-jaksel-harap-banyak-masyarakat-ikut-lelang-rubicon-mario-dandy",
"https://nasional.kompas.com/read/2024/08/29/06510571/pengamat-nilai-wajar-pdi-p-tak-jadi-calonkan-anies",
"https://nasional.kompas.com/read/2024/09/05/13381861/alissa-wahid-paus-fransiskus-ajak-umat-jaga-tali-persaudaraan-antar-agama",
"https://nasional.kompas.com/read/2024/08/18/21510451/survei-smrc-anies-unggul-jika-head-to-head-dengan-ridwan-kamil-pada-pilkada",
"https://megapolitan.kompas.com/read/2024/08/08/19185211/parkir-liar-depan-stasiun-bekasi-bikin-macet-pj-wali-kota-apakah",
"https://nasional.kompas.com/read/2024/08/14/20014481/sebut-ikn-bebas-polusi-zulhas-kalau-di-jakarta-kita-hirup-timbal-dan-bisa",
"https://megapolitan.kompas.com/read/2024/02/26/21241071/seorang-pria-lakukan-percobaan-pembunuhan-di-perkantoran-jatinegara-kejar",
"https://nasional.kompas.com/read/2024/05/23/20053171/tak-ada-jalan-pintas-hasto-politik-harus-belajar-dari-olahraga",
"https://nasional.kompas.com/read/2024/08/06/14044561/soal-cak-imin-bawa-istri-ikut-timwas-haji-mkd-dpr-tak-ada-pelanggaran-hukum",
"https://nasional.kompas.com/read/2024/02/27/21101401/ramai-wacana-hak-angket-menko-polhukam-diharap-bijak-tanggapi-isu-politik",
"https://megapolitan.kompas.com/read/2024/08/22/17340931/2-ruas-tol-depan-gedung-dpr-ri-ditutup-massa-makin-banyak-yang-datang",
"https://nasional.kompas.com/read/2024/08/24/17345931/psi-akui-bantu-kaesang-urus-persyaratan-pilkada-kini-dihentikan-pasca",
"https://nasional.kompas.com/read/2024/08/12/16073641/sambangi-bareskrim-keluarga-wni-yang-disekap-di-myanmar-minta-korban",
"https://nasional.kompas.com/read/2024/08/13/11505261/dirut-jasa-raharja-minta-jajaran-edukasi-masyarakat-tentang-bayar-pajak-dan",
"https://nasional.kompas.com/read/2024/07/30/14553491/tak-setuju-uu-tni-dan-polri-direvisi-megawati-kok-sekarang-disetarakan",
"https://nasional.kompas.com/read/2024/07/28/19091851/minta-ke-menteri-basuki-jokowi-di-kanan-kiri-jalan-tol-ikn-harus-super-hijau",
"https://nasional.kompas.com/read/2024/09/04/15094961/paus-fransiskus-gereja-katolik-ingin-tingkatkan-dialog-antaragama",
"https://megapolitan.kompas.com/read/2024/08/22/11340881/mahasiswa-ui-bersiap-terjun-dalam-aksi-tolak-revisi-uu-pilkada-di-gedung",
"https://megapolitan.kompas.com/read/2024/05/07/20283801/mahasiswa-dikeroyok-di-tangsel-setara-institute-minta-hentikan-narasi",
"https://nasional.kompas.com/read/2024/08/21/20353551/bahlil-ingatkan-prabowo-dilahirkan-golkar-sudah-betul-barang-ini-jadi",
"https://megapolitan.kompas.com/read/2024/02/23/21483301/tiket-ka-jarak-jauh-h-2-lebaran-sudah-terjual-90-persen-per-hari-ini",
"https://megapolitan.kompas.com/read/2024/05/03/20034391/dokter-belum-visum-jenazah-mahasiswa-stip-yang-tewas-akibat-diduga",
"https://megapolitan.kompas.com/read/2024/07/29/15333591/fitri-rahmadani-hilang-hampir-2-pekan-keluarga-kecewa-karena-nilai-polisi",
"https://megapolitan.kompas.com/read/2024/01/18/23523551/dishub-dki-cabut-8741-stick-cone-secara-bertahap-untuk-jamin-keselamatan",
"https://nasional.kompas.com/read/2024/08/29/20463391/brutalitas-aparat-jadi-sorotan-jokowi-diminta-copot-kapolri-dari-jabatannya",
"https://megapolitan.kompas.com/read/2024/08/09/08571351/sebut-anies-mungkin-gagal-dicalonkan-pks-kepastian-dukungan-cagub-jakarta",
"https://megapolitan.kompas.com/read/2024/07/29/15515871/selebgram-medan-tewas-usai-sedot-lemak-klinik-sebut-dokter-yang-menangani",
"https://nasional.kompas.com/read/2024/05/26/22125901/ganjar-pranowo-17-poin-rekomendasi-rakernas-beri-gambaran-sikap-politik-pdip",
"https://megapolitan.kompas.com/read/2024/07/30/15004021/anaknya-dianiaya-orangtua-korban-laporkan-pemilik-daycare-di-depok-ke",
"https://nasional.kompas.com/read/2024/08/22/13582931/kena-lemparan-botol-dari-demonstran-habiburokhman-risiko-wakil-rakyat",
"https://megapolitan.kompas.com/read/2024/08/06/19104221/satu-warga-terluka-usai-bentrokan-di-gambir",
"https://nasional.kompas.com/read/2024/08/21/14482361/uu-pilkada-direvisi-kepala-daerah-hasil-pilkada-diduga-melanggar-mesti-siap",
"https://nasional.kompas.com/read/2024/08/30/18204081/kpk-akan-dalami-kemungkinan-kaesang-dapat-fasilitas-karena-campur-tangan",
"https://nasional.kompas.com/read/2024/05/22/21061021/wwf-2024-jadi-komitmen-dan-aksi-nyata-pertamina-kelola-keberlangsungan-air",
"https://nasional.kompas.com/read/2024/01/07/23193241/prabowo-pastikan-tni-dan-polri-tetap-berada-di-bawah-presiden-jika-terpilih",
"https://nasional.kompas.com/read/2024/06/13/20523491/soal-usung-siapa-di-pilkada-jakarta-nasdem-sebut-anies-dan-tokoh-lain-punya",
"https://megapolitan.kompas.com/read/2024/09/04/16090171/tak-segan-minta-masukan-pramono-anung-sebut-ahok-beri-banyak-legacy-buat",
"https://megapolitan.kompas.com/read/2024/08/05/18455751/tak-dipecat-kepala-smpn-19-hanya-disanksi-disiplin-ringan-terkait-kasus",
"https://nasional.kompas.com/read/2024/08/23/17344401/maju-pilkada-jateng-bersama-taj-yasin-ahmad-luthfi-minta-doa-restu",
"https://nasional.kompas.com/read/2024/08/16/18442881/ridwan-kamil-versus-calon-independen-pengamat-pilkada-jakarta-game-over",
"https://nasional.kompas.com/read/2024/09/02/13460541/pemerintah-siap-gandeng-negara-negara-afrika-tanggulangi-cacar-monyet",
"https://nasional.kompas.com/read/2024/08/14/21075381/kuasa-hukum-sebut-dakwaan-harvey-moeis-rugikan-negara-rp-300-triliun-salah",
"https://nasional.kompas.com/read/2024/09/05/06450071/konflik-pkb-pbnu--pengabaian-politik-dan-pengkhianatan-prinsip",
"https://megapolitan.kompas.com/read/2024/09/03/16303251/komplotan-penipu-beraksi-di-kelapa-gading-sasar-korban-dengan-modus-tukar",
"https://nasional.kompas.com/read/2024/07/31/15320181/dpr-minta-tni-serius-dan-transparan-tangani-dugaan-penganiayaan-pelajar",
"https://nasional.kompas.com/read/2024/01/01/20350511/sambangi-rumah-kakeknya-anies-teringat-perjuangan-abdurrahman-baswedan",
"https://nasional.kompas.com/read/2024/03/17/20360841/imparsial-nilai-pp-manajemen-asn-akan-perluas-masuknya-tni-polri-ke-ranah",
"https://nasional.kompas.com/read/2024/05/01/23001581/prediksi-soal-kabinet-prabowo-gibran-menteri-triumvirat-tak-diberi-ke-parpol",
"https://lifestyle.kompas.com/read/2025/08/31/100000720/waspadai-kelelahan-mental-akibat-kebanyakan-berita-negatif-",
]

results = [scrape_kompas_article(u) for u in urls_to_test if scrape_kompas_article(u)]
df = pd.DataFrame(results)

from IPython.display import display
display(df)

,Judul,Isi (100 kata),Kategori
0,Jokowi Kenakan Pakaian Adat Betawi di Sidang T...,"JAKARTA, KOMPAS.com- Presiden Joko Widodo mema...",Nasional
1,KPU Tegaskan Pemilih Tak Terdaftar di DPT Bisa...,"JAKARTA, KOMPAS.com- Komisi Pemilihan Umum (KP...",Nasional
2,Warga Sebut Ada Benda Serupa Jimat pada Mayat ...,"TANGERANG SELATAN, KOMPAS.com- Seutas tali ber...",Megapolitan
3,Polisi Menganiaya Mereka yang Cinta Damai dan ...,"JAKARTA, KOMPAS.com- Sejumlah massa mendapat k...",Megapolitan
4,"Airlangga Hartarto Mundur dari Ketum, Golkar B...","JAKARTA, KOMPAS.com- Airlangga Hartarto secara...",Nasional
...,...,...,...
124,DPR Minta TNI Serius dan Transparan Tangani Du...,"JAKARTA, KOMPAS.com- Ketua Komisi I DPR RI Meu...",Nasional
125,"Sambangi Rumah Kakeknya, Anies Teringat Perjua...","YOGYAKARTA, KOMPAS.com- Calon presiden nomor u...",Nasional
126,Imparsial Nilai PP Manajemen ASN Akan Perluas ...,"JAKARTA, KOMPAS.com- Imparsial menilai, penges...",Nasional
127,Prediksi soal Kabinet Prabowo-Gibran: Menteri ...,"JAKARTA, KOMPAS.com- Direktur Eksekutif Poltra...",Nasional
